In [5]:
import kagglehub
import os
import shutil
from sklearn.model_selection import train_test_split

# =========================
# DOWNLOAD DATASETS
# =========================

dataset1 = kagglehub.dataset_download(
    "vipoooool/new-plant-diseases-dataset"
)

dataset2 = kagglehub.dataset_download(
    "kamal01/top-agriculture-crop-disease"
)

print("Dataset 1:", dataset1)
print("Dataset 2:", dataset2)

# =========================
# FINAL DATASET PATHS
# =========================

FINAL_DATASET = "/content/final_dataset"

TRAIN_DIR = os.path.join(FINAL_DATASET, "train")
VALID_DIR = os.path.join(FINAL_DATASET, "valid")

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VALID_DIR, exist_ok=True)

# =========================
# DATASET 1 PATHS
# =========================

dataset1_train = os.path.join(
    dataset1,
    "New Plant Diseases Dataset(Augmented)",
    "New Plant Diseases Dataset(Augmented)",
    "train"
)

dataset1_valid = os.path.join(
    dataset1,
    "New Plant Diseases Dataset(Augmented)",
    "New Plant Diseases Dataset(Augmented)",
    "valid"
)

# =========================
# COPY DATASET 1
# =========================

print("Copying Dataset 1...")

# TRAIN
for class_name in os.listdir(dataset1_train):

    src = os.path.join(dataset1_train, class_name)
    dst = os.path.join(TRAIN_DIR, class_name)

    shutil.copytree(src, dst, dirs_exist_ok=True)

# VALID
for class_name in os.listdir(dataset1_valid):

    src = os.path.join(dataset1_valid, class_name)
    dst = os.path.join(VALID_DIR, class_name)

    shutil.copytree(src, dst, dirs_exist_ok=True)

print("Dataset 1 copied!")

# =========================
# DATASET 2 PATH
# =========================

crop_dataset = os.path.join(
    dataset2,
    "Crop Diseases"
)

# =========================
# SPLIT DATASET 2
# =========================

print("Processing Dataset 2...")

for class_name in os.listdir(crop_dataset):

    class_path = os.path.join(crop_dataset, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)

    train_imgs, valid_imgs = train_test_split(
        images,
        test_size=0.2,
        random_state=42
    )

    # TRAIN
    train_class_path = os.path.join(
        TRAIN_DIR,
        class_name
    )

    os.makedirs(train_class_path, exist_ok=True)

    for img in train_imgs:

        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(train_class_path, img)
        )

    # VALID
    valid_class_path = os.path.join(
        VALID_DIR,
        class_name
    )

    os.makedirs(valid_class_path, exist_ok=True)

    for img in valid_imgs:

        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(valid_class_path, img)
        )

print("Dataset 2 processed!")


print("FINAL DATASET READY!")
print("Train Folder:", TRAIN_DIR)
print("Valid Folder:", VALID_DIR)

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Using Colab cache for faster access to the 'top-agriculture-crop-disease' dataset.
Dataset 1: /kaggle/input/new-plant-diseases-dataset
Dataset 2: /kaggle/input/top-agriculture-crop-disease
Copying Dataset 1...


KeyboardInterrupt: 

In [2]:
# ==========================================
# COMPLETE LEAF DISEASE TRAIN + PREDICT CODE
# ==========================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

# ==========================================
# DATASET PATHS
# ==========================================

TRAIN_DIR = "/content/final_dataset/train"
VALID_DIR = "/content/final_dataset/valid"

# ==========================================
# IMAGE SETTINGS
# ==========================================

IMG_SIZE = 224
BATCH_SIZE = 32

# ==========================================
# LOAD DATASET
# ==========================================

train_datagen = ImageDataGenerator(
    rescale=1./255
)

valid_datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# ==========================================
# CLASS NAMES
# ==========================================

class_names = list(train_data.class_indices.keys())

print("Classes:")
print(class_names)

# ==========================================
# BUILD MODEL
# ==========================================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256, activation='relu'),

    Dropout(0.3),

    Dense(
        train_data.num_classes,
        activation='softmax'
    )
])

# ==========================================
# COMPILE MODEL
# ==========================================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# TRAIN MODEL
# ==========================================

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=5
)

# ==========================================
# SAVE MODEL
# ==========================================

model.save("leaf_disease_model.h5")

print("✅ Model Saved Successfully!")


Found 80949 images belonging to 55 classes.
Found 20242 images belonging to 55 classes.
Classes:
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Corn___Common_Rust', 'Corn___Gray_Leaf_Spot', 'Corn___Healthy', 'Corn___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_Blight', 'Potato___Early_blight', 'Potato___Healthy', 'Potato___Late_Blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Rice___Brown_Spot', 'Rice___Health

✅ Model Saved Successfully!


In [3]:
from google.colab import files

files.download("leaf_disease_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>